<div dir="rtl">

# الدرس 39: التدريب الجيد: المحسِّنات ومعدل التعلّم

دفتر الكود لهذا الدرس. شغّل الخلايا بالترتيب من الأعلى.

</div>

<div dir="rtl">

### الخطوات المحلولة، بمحسّنات باي تورش

</div>

In [ ]:
import math
import numpy as np
import torch
from torch import nn

def two_steps(momentum):
    w = torch.tensor(1.0, requires_grad=True)
    opt = torch.optim.SGD([w], lr=0.1, momentum=momentum)
    for _ in range(2):
        loss = w ** 2
        opt.zero_grad(); loss.backward(); opt.step()
    return round(w.item(), 4)

print("plain SGD after 2 steps:", two_steps(0.0))
print("momentum  after 2 steps:", two_steps(0.9))

p = torch.tensor([0.0, 0.0], requires_grad=True)
opt = torch.optim.Adam([p], lr=0.001)
(100 * p[0] + 0.01 * p[1]).backward()        # gradients: 100 and 0.01
opt.step()
print("Adam first step:", p.detach().numpy())

<div dir="rtl">

### المحسّنات على الحلزون

</div>

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

def make_spiral(n_per_class=100, n_classes=3, noise=0.2, seed=0):
    rng = np.random.default_rng(seed)
    X, y = [], []
    for k in range(n_classes):
        r = np.linspace(0.0, 1.0, n_per_class)
        t = np.linspace(4 * k, 4 * (k + 1), n_per_class) + rng.normal(0, noise, n_per_class)
        X.append(np.column_stack([r * np.sin(t), r * np.cos(t)]))
        y.append(np.full(n_per_class, k))
    return np.vstack(X), np.concatenate(y)

X, y = make_spiral()
idx = np.random.default_rng(1).permutation(len(X))
X_tr = torch.tensor(X[idx[:240]], dtype=torch.float32)
y_tr = torch.tensor(y[idx[:240]])
X_val = torch.tensor(X[idx[240:]], dtype=torch.float32)
y_val = torch.tensor(y[idx[240:]])
loss_fn = nn.CrossEntropyLoss()

def make_model():
    torch.manual_seed(0)                           # same initial weights every time
    return nn.Sequential(nn.Linear(2, 64), nn.ReLU(), nn.Linear(64, 3))

def train(make_opt, epochs=100):
    model = make_model()
    opt = make_opt(model.parameters())
    loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=32, shuffle=True,
                        generator=torch.Generator().manual_seed(0))
    history = []
    for _ in range(epochs):
        model.train()
        for xb, yb in loader:
            loss = loss_fn(model(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            history.append(loss_fn(model(X_tr), y_tr).item())
    with torch.no_grad():
        acc = (model(X_val).argmax(1) == y_val).float().mean().item()
    return history, acc

In [ ]:
optimizers = {
    "SGD, lr=0.1":      lambda p: torch.optim.SGD(p, lr=0.1),
    "Momentum, lr=0.1": lambda p: torch.optim.SGD(p, lr=0.1, momentum=0.9),
    "Adam, lr=0.01":    lambda p: torch.optim.Adam(p, lr=0.01),
}
histories = {}
for name, make_opt in optimizers.items():
    histories[name], acc = train(make_opt)
    print(f"{name:17}: loss after 10 epochs {histories[name][9]:.3f}, "
          f"after 100 {histories[name][-1]:.3f}, val acc {acc:.3f}")

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(7, 3.4))
for name, h in histories.items():
    plt.plot(range(1, len(h) + 1), h, label=name)
plt.yscale("log"); plt.xlabel("epoch"); plt.ylabel("train loss")
plt.legend(); plt.grid(True, which="both", alpha=0.4)
plt.show()

<div dir="rtl">

### البحث عن معدل التعلّم

</div>

In [ ]:
lrs = [1e-4, 1e-3, 1e-2, 3e-2, 1e-1, 3e-1, 1.0, 3.0, 10.0]
final = []
for lr in lrs:
    h, acc = train(lambda p: torch.optim.Adam(p, lr=lr), epochs=30)
    final.append(h[-1])
    print(f"lr = {lr:<6g}: train loss {h[-1]:.3f}, val acc {acc:.3f}")

In [ ]:
plt.figure(figsize=(6, 3.2))
plt.plot(lrs, final, "o-")
plt.xscale("log"); plt.xlabel("learning rate"); plt.ylabel("loss after 30 epochs")
plt.grid(True, which="both", alpha=0.4)
plt.show()

<div dir="rtl">

### جداول معدل التعلّم

</div>

In [ ]:
steps, warmup, lr_max = 1000, 100, 0.1

def warmup_cosine(t):
    if t < warmup:
        return (t + 1) / warmup                              # linear warmup
    progress = (t - warmup) / (steps - warmup)
    return 0.5 * (1 + math.cos(math.pi * progress))          # cosine decay

def lr_curve(make_scheduler):
    opt = torch.optim.SGD([torch.zeros(1, requires_grad=True)], lr=lr_max)
    sched = make_scheduler(opt)
    values = []
    for _ in range(steps):
        values.append(opt.param_groups[0]["lr"])
        opt.step(); sched.step()                           # once per training step
    return values

curves = {
    "step decay": lr_curve(lambda o: torch.optim.lr_scheduler.StepLR(o, step_size=400, gamma=0.1)),
    "cosine": lr_curve(lambda o: torch.optim.lr_scheduler.CosineAnnealingLR(o, T_max=steps)),
    "warmup + cosine": lr_curve(lambda o: torch.optim.lr_scheduler.LambdaLR(o, warmup_cosine)),
}
for name, c in curves.items():
    print(f"{name:16}: step 0 → {c[0]:.4f}, step 50 → {c[50]:.4f}, step 500 → {c[500]:.4f}, step 999 → {c[999]:.5f}")

In [ ]:
plt.figure(figsize=(7, 3))
for name, c in curves.items():
    plt.plot(c, label=name, lw=2)
plt.xlabel("training step"); plt.ylabel("learning rate"); plt.legend(); plt.grid(True, alpha=0.4)
plt.show()

<div dir="rtl">

### الأوزان الابتدائية: عبر عشرين طبقة

</div>

In [ ]:
torch.manual_seed(0)
x = torch.randn(1000, 256)
for name, std in [("std 0.01", 0.01), ("std 0.2", 0.2), ("He", math.sqrt(2 / 256))]:
    torch.manual_seed(1)
    h, stds = x, []
    for layer in range(20):
        h = torch.relu(h @ (torch.randn(256, 256) * std))
        stds.append(h.std().item())
    print(f"{name:9}: layer 1 {stds[0]:.1e} | layer 5 {stds[4]:.1e} | "
          f"layer 10 {stds[9]:.1e} | layer 20 {stds[19]:.1e}")

<div dir="rtl">

### التماثل: عصبونات متطابقة إلى الأبد

</div>

In [ ]:
sym = nn.Sequential(nn.Linear(2, 4), nn.ReLU(), nn.Linear(4, 3))
with torch.no_grad():
    for layer in (sym[0], sym[2]):
        layer.weight.fill_(0.5)                # every neuron starts identical
        layer.bias.zero_()
opt = torch.optim.SGD(sym.parameters(), lr=0.5)
for _ in range(200):
    loss = loss_fn(sym(X_tr), y_tr)
    opt.zero_grad(); loss.backward(); opt.step()
print(sym[0].weight.detach())

<div dir="rtl">

### تطبيع الطبقة

</div>

In [ ]:
ln = nn.LayerNorm(3)
print(ln(torch.tensor([[2.0, 4.0, 6.0]])).detach())
print("gamma:", ln.weight.data, " beta:", ln.bias.data)

<div dir="rtl">

## تمرين: جدول الإحماء بيدك، ثم بالكود

جدول: إحماء خطي لأول 100 خطوة، ثم جيب التمام حتى الخطوة 1000. والقيمة العظمى 0.1. نستخدم نفس دالة الإحماء في الكود أعلاه.

1. احسب معدل التعلّم بيدك عند الخطوات: 0، و 49، و 99، و 550، و 999.
2. تحقق من إجاباتك بالكود.

</div>

In [ ]:
# اكتب حلّك هنا
